In [1]:
# ============================================================
#  08 POI Demand Layers
#  数据源: 00/ 本地 POI 数据 (2026.01, ~490K 条, 无需 API)
#  输出: POI 点位 + 网格化需求密度
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
from shapely.prepared import prep as shapely_prep
import warnings
warnings.filterwarnings("ignore")

OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

BOUNDARY = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")

# 00 本地 POI 文件 → demand 分类
POI_BASE = Path("../00/02-POI&AOI/02-POI&AOI/1-POI/2026.01/SHP/分类")
POI_FILES = {
    "深圳市-餐饮服务.shp":     "food",
    "深圳市-购物服务.shp":     "retail",
    "深圳市-医疗保健服务.shp":  "medical",
    "深圳市-科教文化服务.shp":  "education",
    "深圳市-商务住宅.shp":     "office",
    "深圳市-体育休闲服务.shp":  "leisure",
    "深圳市-风景名胜.shp":     "scenic",
    "深圳市-生活服务.shp":     "service",
}

shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union
sz_prep = shapely_prep(shenzhen_geom)

print(f"POI source: {POI_BASE}")
print(f"Categories: {len(POI_FILES)}")

POI source: ../00/02-POI&AOI/02-POI&AOI/1-POI/2026.01/SHP/分类
Categories: 8


In [2]:
# ============================================================
#  1. 读取本地 POI 数据 (00/2026.01, 秒级完成)
# ============================================================

all_frames = []
for filename, demand_group in POI_FILES.items():
    path = POI_BASE / filename
    if not path.exists():
        print(f"  MISSING: {path}")
        continue
    gdf = gpd.read_file(path)
    gdf["poi_type"] = demand_group
    keep = ["name", "address", "poi_type", "geometry"]
    keep = [c for c in keep if c in gdf.columns]
    all_frames.append(gdf[keep])
    print(f"  {demand_group:12s}  {len(gdf):>9,}  <- {filename}")

poi_raw = pd.concat(all_frames, ignore_index=True)
print(f"\nTotal raw POIs: {len(poi_raw):,}")

  food            108,953  <- 深圳市-餐饮服务.shp
  retail          162,881  <- 深圳市-购物服务.shp
  medical          26,369  <- 深圳市-医疗保健服务.shp
  education        24,709  <- 深圳市-科教文化服务.shp
  office           35,892  <- 深圳市-商务住宅.shp
  leisure          24,409  <- 深圳市-体育休闲服务.shp
  scenic            5,875  <- 深圳市-风景名胜.shp
  service         101,544  <- 深圳市-生活服务.shp

Total raw POIs: 490,632


In [3]:
# ============================================================
#  2. 清洗去重 + 裁剪
# ============================================================

poi_gdf = gpd.GeoDataFrame(poi_raw, crs=4326)
print(f"Raw: {len(poi_gdf):,}")

# 去重: 相同名称 + 相近坐标
poi_gdf["_lon"] = poi_gdf.geometry.x.round(5)
poi_gdf["_lat"] = poi_gdf.geometry.y.round(5)
poi_gdf = poi_gdf.drop_duplicates(subset=["name", "_lon", "_lat"]).drop(columns=["_lon", "_lat"])
print(f"After dedup: {len(poi_gdf):,}")

# 裁剪到深圳边界
poi_gdf = gpd.clip(poi_gdf, shenzhen_geom)
print(f"After clip: {len(poi_gdf):,}")

# 添加 lon/lat 列 (供下游使用)
poi_gdf["lon"] = poi_gdf.geometry.x
poi_gdf["lat"] = poi_gdf.geometry.y

# demand_group = poi_type 本身 (已经是大类)
poi_gdf["demand_group"] = poi_gdf["poi_type"]

print(f"\n=== POI by type ===")
print(poi_gdf["poi_type"].value_counts().to_string())

Raw: 490,632
After dedup: 478,437
After clip: 478,246

=== POI by type ===
poi_type
retail       156754
food         107650
service       99690
office        34841
medical       26044
education     23966
leisure       23534
scenic         5767


In [4]:
# ============================================================
#  3. 网格化需求密度 + demand_pressure
#  复用 06 Buildings 的网格大小 (0.005° ≈ 550m)
# ============================================================

GRID_SIZE = 0.005

minx, miny, maxx, maxy = shenzhen_geom.bounds

grid_cells = []
gid = 0
x = minx
while x < maxx:
    y = miny
    while y < maxy:
        cell = box(x, y, x + GRID_SIZE, y + GRID_SIZE)
        if sz_prep.intersects(cell):
            grid_cells.append({"grid_id": gid, "geometry": cell})
            gid += 1
        y += GRID_SIZE
    x += GRID_SIZE

grid = gpd.GeoDataFrame(grid_cells, crs=4326)
print(f"Grid: {len(grid):,} cells")

# 空间连接 POI → 网格
joined = gpd.sjoin(poi_gdf, grid[["grid_id", "geometry"]], how="inner", predicate="within")

# 按需求大类分组统计
pivot = joined.groupby(["grid_id", "demand_group"]).size().unstack(fill_value=0)
pivot.columns = [f"{c}_count" for c in pivot.columns]
pivot["poi_total"] = pivot.sum(axis=1)
pivot = pivot.reset_index()

# 合并回网格
grid_demand = grid.merge(pivot, on="grid_id", how="left").fillna(0)

# demand_pressure: 加权综合需求指数
WEIGHTS = {
    "food_count": 0.25,
    "retail_count": 0.20,
    "medical_count": 0.15,
    "office_count": 0.15,
    "education_count": 0.05,
    "service_count": 0.05,
    "leisure_count": 0.05,
    "scenic_count": 0.05,
    "residential_count": 0.05,
}

grid_demand["demand_pressure"] = 0
for col, w in WEIGHTS.items():
    if col in grid_demand.columns:
        grid_demand["demand_pressure"] += w * grid_demand[col]

# 归一化到 [0, 1]
dp = grid_demand["demand_pressure"]
dp_max = dp.max()
if dp_max > 0:
    grid_demand["demand_pressure_norm"] = dp / dp_max

non_empty = (grid_demand["poi_total"] > 0).sum()
print(f"Grids with POIs: {non_empty:,} / {len(grid_demand):,}")
print(f"Demand pressure: max={dp.max():.1f}, mean(non-empty)={dp[dp > 0].mean():.1f}")

Grid: 7,392 cells
Grids with POIs: 4,769 / 7,392
Demand pressure: max=305.0, mean(non-empty)=15.7


In [5]:
# ============================================================
#  4. 保存 + 统计摘要
# ============================================================

# POI 点位
poi_out = poi_gdf[["name", "poi_type", "demand_group", "address", "lon", "lat", "geometry"]]
poi_out.to_file(OUT / "sz_poi.gpkg", driver="GPKG")
print(f"POI      -> data_out/sz_poi.gpkg          ({len(poi_out):,} rows)")

# 需求网格
grid_demand.to_file(OUT / "sz_demand_grid.gpkg", driver="GPKG")
print(f"Grid     -> data_out/sz_demand_grid.gpkg   ({len(grid_demand):,} cells)")

# 统计摘要
print("\n=== POI Summary ===")
print(f"Total POIs: {len(poi_out):,}")
print(poi_out["poi_type"].value_counts().to_string())

print("\n=== Demand Grid Summary (non-empty) ===")
g = grid_demand[grid_demand["poi_total"] > 0]
for col in [c for c in grid_demand.columns if c.endswith("_count")] + ["demand_pressure"]:
    if col in g.columns:
        print(f"  {col:25s}  mean={g[col].mean():8.1f}  max={g[col].max():8.0f}")

POI      -> data_out/sz_poi.gpkg          (478,246 rows)
Grid     -> data_out/sz_demand_grid.gpkg   (7,392 cells)

=== POI Summary ===
Total POIs: 478,246
poi_type
retail       156754
food         107650
service       99690
office        34841
medical       26044
education     23966
leisure       23534
scenic         5767

=== Demand Grid Summary (non-empty) ===
  education_count            mean=     5.0  max=      90
  food_count                 mean=    22.6  max=     321
  leisure_count              mean=     4.9  max=     148
  medical_count              mean=     5.5  max=     105
  office_count               mean=     7.3  max=      67
  retail_count               mean=    32.9  max=    1000
  scenic_count               mean=     1.2  max=      90
  service_count              mean=    20.9  max=     363
  demand_pressure            mean=    15.7  max=     305
